# Price the architectures

The same workload can be run four ways. They cost different amounts of
money, and the ranking is not obvious from first principles — which is
why it is worth computing.

The point of this notebook is that **the comparison is a function of its
inputs, not a fixed ranking of technologies.** Change how long the
pipeline runs, or how much of the table the query reads, and the winner
changes.

## 0. Setup

Put the course's pure core modules on the path. No installation
and no cloud SDK — that is the whole point of the split.

In [ ]:
import sys
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project" / "core").exists():
    COURSE_ROOT = Path.cwd().parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))

SAMPLE_DIR = COURSE_ROOT / "sample_data"


## 1. The four options

A streaming pipeline, a Spark cluster, a partitioned warehouse scan
and an unpartitioned one — all doing the same logical work.

In [ ]:
from cost_model import compare_architectures, format_comparison

options = compare_architectures(scanned_tb=1.0, partition_factor=0.01)
print(format_comparison(options))

## 2. Why the two zero-cost rows are zero

They are zero because `streaming_hours` defaults to `0`. The
Dataflow and Dataproc lines are priced per **vCPU-hour**; no hours
were requested, so nothing was billed.

Never read a `$0.00` streaming row as *free*. Read it as *the hours
were not counted*.

In [ ]:
options = compare_architectures(
    scanned_tb=1.0, partition_factor=0.01, streaming_hours=24
)
print(format_comparison(options))

## 3. The partitioning multiplier

A hundred-fold reduction from the table layout alone, with no change
to the query. This is the single highest-return optimisation in the
whole course.

In [ ]:
from cost_model import BIGQUERY_ONDEMAND_PER_TB

full = BIGQUERY_ONDEMAND_PER_TB * 1.0
pruned = BIGQUERY_ONDEMAND_PER_TB * 1.0 * 0.01
print(f'full scan:    ${full:.4f}')
print(f'pruned scan:  ${pruned:.4f}')
print(f'ratio:        {full / pruned:.0f}x')

## 4. The demo project's monthly bill

The simulation prices every service the demo project uses. Note
that the continuously-running services dominate — which is the
opposite of what people expect when they think *BigQuery is
expensive*.

In [ ]:
from platform_sim import build_demo_project

project = build_demo_project()
costs = project.estimate_monthly_cost()
total = 0.0
for service, cost in sorted(costs.items(), key=lambda kv: -kv[1]):
    print(f'{service:<24} ${cost:>10.4f}')
    total += cost
print('TOTAL'.ljust(24) + ' $' + format(total, '>10.4f'))